In [ ]:
""" Created on November 13, 2023 // @author: Sarah Shi """

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


import mineralML as mm

%matplotlib inline
%config InlineBackend.figure_format = 'png'

# Synthetic Mineral Generator

This notebook shows **how the synthetic mineral generator in mineralML works**, with an example CSV for groundtruthing: `training_hundred.csv`. This is a three step process: 
1. Load and prepare data for analysis
2. Define endmembers and generator settings (e.g., oxygen_basis, mixing distribution/parameters, minor elements, noise scales).
3. Generate synthetic compositions and evaluate them (convert to oxide wt% and cations; optionally use compare_distributions to compare against the natural dataset).

We loaded in the ``mineralML`` Python package as ``mm``. ``mineralML`` has trained machine learning models for classifying minerals. This implementation aims to get your electron microprobe or quantitative EDS compositions classified and processed. We remove some degrees of freedom to simplify the process as much as possible. The minerals considered for this study include: Amphibole, Apatite, Biotite, Calcite, Chlorite, Epidote, Feldspar (KFeldspar and Plagioclase), Garnet, Glass, Kalsilite, Leucite, Melilite, Muscovite, Nepheline, Olivine, Pyroxene (Clinopyroxene and Orthopyroxene), Quartz, Rhombohedral_Oxides (Hematite-Ilmenite), Rutile, Serpentine, Spinels (Magnetite-Spinel), Titanite, Tourmaline, and Zircon. 

One CSV file containing your electron microprobe analyses in oxide weight percentages is necessary. Find an example [here](https://github.com/sarahshi/mineralML/blob/main/docs/examples/training_hundred.csv). The necessary oxides are $SiO_2$, $TiO_2$, $Al_2O_3$, $FeO_t$, $MnO$, $MgO$, $CaO$, $Na_2O$, $K_2O$, $Cr_2O_3$, and $P_2O_5$. For the oxides not analyzed for specific minerals, the preprocessing will fill in the nan values as 0. 

## Load and prepare data for groundtruthing

In [ ]:
# Read in your dataframe of mineral data, called training_hundred.csv. 
# Prepare the dataframe by removing rows with too many NaNs, and filling in zeros. 

df_load = mm.load_df('synth_groundtruth.csv')

In [ ]:
# Examine the prepared dataframe

display(df_load.head())

# Olivine

Let’s apply the generator to olivine as a simple binary solid solution between forsterite (Mg₂SiO₄) and fayalite (Fe₂SiO₄). We understand olivine systematics quite well, so we can test this before applying this to a more complex system. We will keep everything on a 4-oxygen basis, add small amounts of Ca and Mn as minors, and then check that the synthetic cloud matches the natural data. The steps are as follow: 

1. Define endmembers (4 oxygen basis). Use cation counts per formula unit; iron as total cations (Fe2t) so the framework can convert to FeOt downstream.
2. Specify minor elements (optional but realistic).
3. Instantiate the generator.
4. Generate synthetic compositions.
5. Compute sites/derived components.
6. Compare synthetic vs natural distributions.
7. Plot paired violin distributions for cations (and matching oxides if present). Report KS statistics (ks_stat, p_value) plus means/stds. Lower ks_stat / higher p_value ⇒ better match.

Gotchas:
- Keep iron conventions straight: Fe2t (cations) align with FeOt (oxide). Don’t mix FeO/Fe₂O₃ with FeOt in the same row.

In [ ]:
# Pull natural data 
df_ol_natural = df_load[df_load["Mineral"]=="Olivine"]
ol_calc_natural = mm.OlivineCalculator(df_ol_natural)
ol_comp_natural = ol_calc_natural.calculate_components()

# Define endmembers 
ol_endmembers = {
    # Forsterite: Mg₂SiO₄
    'Fo': {'Mg': 2, 'Si': 1, 'O': 4},
    # Fayalite: Fe₂SiO₄
    'Fa': {'Fe2t': 2, 'Si': 1, 'O': 4}
}

# Specify minor elements
ol_minors = {
    'Ca': {'distribution': 'exponential', 'scale': 0.01, 'max_fraction': 0.01},
    'Mn': {'distribution': 'exponential', 'scale': 0.01, 'max_fraction': 0.01}
}

# Instantiate generator
ol_gen = mm.SolidSolutionGenerator(
    endmembers=ol_endmembers,
    oxygen_basis=4,
    element_noise_scale=0.025,
    min_site_fraction=0.2,
    minor_elements=ol_minors,
    mixing_dist='beta',
    mixing_params={'a': 1, 'b': 1}
)

# Generate samples, use the olivine calculator to calculate site allocations, etc. 
df_ol = ol_gen.generate(1000)
ol_calc_synth = mm.OlivineCalculator(df_ol)
ol_comp_synth = ol_calc_synth.calculate_components()
display(ol_comp_synth)

# Calculate and compare the distributions of the output data
stats_ol = ol_gen.compare_distributions(base_df=ol_comp_natural, synth_df=ol_comp_synth, suptitle="Olivine")
display(stats_ol)

# Scatter‐plot comparing base vs. synthetic oxide proportions
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
ax[0].scatter(ol_comp_natural["FeOt"], ol_comp_natural["MgO"], s=20, c="g", lw=0.25, ec='k')
ax[0].scatter(ol_comp_synth["FeOt"], ol_comp_synth["MgO"], s=20, c="r", lw=0.5, ec='k')
ax[0].set_xlabel("FeO")
ax[0].set_ylabel("MgO")

ax[1].scatter(ol_comp_natural["SiO2"], ol_comp_natural["MgO"], s=20, c="g", lw=0.25, ec='k')
ax[1].scatter(ol_comp_synth["SiO2"], ol_comp_synth["MgO"], s=20, c="r", lw=0.5, ec='k')
ax[1].set_xlabel("SiO2")
ax[1].set_ylabel("MgO")

ax[2].scatter(ol_comp_natural["XFo"], ol_comp_natural["M_site_expanded"], s=20, c="g", lw=0.25, ec='k', label="Natural")
ax[2].scatter(ol_comp_synth["XFo"], ol_comp_synth["M_site_expanded"], s=20, c="r", lw=0.25, ec='k', label="Synthetic")
ax[2].set_xlabel("XFo (Mg/(Mg+Fe))")
ax[2].set_ylabel("M-site Expanded")
ax[2].legend()
plt.tight_layout()

# Feldspar

Let's double check that this generator works, and apply this to plagioclase as a simple binary solid solution between albite (NaAlSi₃O₈) and anorthite (CaAl₂Si₂O₈). We understand plagioclase feldspar systematics quite well, so we can test this before applying this to a more complex system. We will have an 8-oxygen basis, add small amounts of K as a minor element, and then check that the synthetic cloud matches the natural data. The steps are as above.

In [ ]:
# Pull natural data 
df_plag_natural = df_load[df_load["Mineral"]=="Plagioclase"]
plag_calc_natural = mm.FeldsparCalculator(df_plag_natural)
plag_comp_natural = plag_calc_natural.calculate_components()

# Define endmembers 
plag_endmembers = {
    # Albite: NaAlSi₃O₈
    'Ab': {'Na': 1, 'Al': 1, 'Si': 3, 'O': 8},
    # Anorthite: CaAl₂Si₂O₈
    'An': {'Ca': 1, 'Al': 2, 'Si': 2, 'O': 8},
}

# Specify minor elements
plag_minors = {'K': {'distribution': 'exponential', 'scale': 0.01, 'max_fraction': 0.02}}

# Instantiate generator
plag_gen = mm.SolidSolutionGenerator(
    endmembers=plag_endmembers,
    oxygen_basis=8,
    element_noise_scale=0.05,
    min_site_fraction=0.2,
    minor_elements=plag_minors,
    mixing_dist='beta',
    mixing_params={'a': 2, 'b': 2}
)

# Generate samples
df_plag = plag_gen.generate(1000)
plag_calc_synth = mm.FeldsparCalculator(df_plag)
plag_comp_synth = plag_calc_synth.calculate_components()
display(plag_comp_synth)

# Calculate and compare the distributions of the output data
stats_pl = plag_gen.compare_distributions(base_df=plag_comp_natural, synth_df=plag_comp_synth, suptitle="Plagioclase")
display(stats_pl)

# Scatter‐plot comparing base vs. synthetic oxide proportions
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
ax[0].scatter(plag_comp_natural["Na2O"], plag_comp_natural["CaO"], s=20, c="g", lw=0.25, ec='k')
ax[0].scatter(plag_comp_synth["Na2O"], plag_comp_synth["CaO"], s=20, c="r", lw=0.25, ec='k')
ax[0].set_xlabel("Na2O")
ax[0].set_ylabel("CaO")

ax[1].scatter(plag_comp_natural["Al2O3"], plag_comp_natural["SiO2"], s=20, c="g", lw=0.25, ec='k')
ax[1].scatter(plag_comp_synth["Al2O3"], plag_comp_synth["SiO2"], s=20, c="r", lw=0.25, ec='k')
ax[1].set_xlabel("Al2O3")
ax[1].set_ylabel("SiO2")

ax[2].scatter(plag_comp_natural["An"], plag_comp_natural["Ab"], s=20, c="g", lw=0.25, ec='k', label="Natural")
ax[2].scatter(plag_comp_synth["An"], plag_comp_synth["Ab"], s=20, c="r", lw=0.25, ec='k', label="Synthetic")
ax[2].set_xlabel("An (Ca/(Ca+Na))")
ax[2].set_ylabel("Ab (Na/(Ca+Na))")
ax[2].legend()
plt.tight_layout()


# Kalsilite

Success, this ``SyntheticMineralGenerator`` works with familiar solid solution minerals. Let's test wonkier (less common) minerals, such as kalsilite. Kalsilite is a feldspathoid mineral that share the tridymite framework. There is a Na-K exhange between nepheline and kalsilite. We will have a 4-oxygen basis and  check that the synthetic cloud matches the natural data. The steps are as above.

In [ ]:
# Pull natural data 
df_ks_natural = df_load[df_load["Mineral"]=="Kalsilite"]
ks_calc_natural = mm.KalsiliteCalculator(df_ks_natural)
ks_comp_natural = ks_calc_natural.calculate_components()

# Define endmembers 
ks_endmembers = {
    # Kalsilite K[AlSiO₄]
    "Ks": {"K":  1, "Al": 1, "Si": 1, "O": 4},
    # Nepheline Na[AlSiO₄] ~ simplification
    "Ne": {"Na": 1, "Al": 1, "Si": 1, "O": 4}
}

# Specify minor elements
ks_minors = {} # no minors for pure K[AlSiO₄]-Na[AlSiO₄]

# Instantiate generator
gen_ks = mm.SolidSolutionGenerator(
    endmembers = ks_endmembers,
    oxygen_basis = 4,
    minor_elements = ks_minors,
    element_noise_scale = 0.02,
    min_site_fraction = 0.2,
    mixing_dist = "beta",
    mixing_params = {"a": 1, "b": 200},
)

# Generate samples, use the kalsilite calculator to calculate site allocations, etc. 
df_ks = gen_ks.generate(n_samples=500)
ks_calc_synth = mm.KalsiliteCalculator(df_ks)
ks_comp_synth = ks_calc_synth.calculate_components()
ks_comp_synth['Mineral'] = 'Kalsilite'
display(ks_comp_synth)

# Calculate and compare the distributions of the output data
stats_ks = gen_ks.compare_distributions(base_df=ks_comp_natural, synth_df=ks_comp_synth, suptitle="Kalsilite")
display(stats_ks)

fig, ax = plt.subplots(1, 4, figsize = (20, 5))
ax = ax.flatten()
ax[0].scatter(ks_comp_natural['Cation_Sum'], ks_comp_natural['Cation_Sum'], s=20, c="g", lw=0.25, ec='k')
ax[0].scatter(ks_comp_synth['Cation_Sum'], ks_comp_synth['Cation_Sum'], s=20, c="r", lw=0.25, ec='k')
ax[0].set_xlabel('Cation_Sum')
ax[0].set_ylabel('Cation_Sum')
ax[1].scatter(ks_comp_natural['A_B_site'], ks_comp_natural['T_site'], s=20, c="g", lw=0.25, ec='k')
ax[1].scatter(ks_comp_synth['A_B_site'], ks_comp_synth['T_site'], s=20, c="g", lw=0.25, ec='r')
ax[1].set_xlabel('A_B_site (K+Na)')
ax[1].set_ylabel('T_site')
ax[2].scatter(ks_comp_natural['K2O'], ks_comp_natural['Na2O'], s=20, c="g", lw=0.25, ec='k')
ax[2].scatter(ks_comp_synth['K2O'], ks_comp_synth['Na2O'], s=20, c="r", lw=0.25, ec='k')
ax[2].set_xlabel('K2O')
ax[2].set_ylabel('Na2O')
ax[3].scatter(ks_comp_natural['SiO2'], ks_comp_natural['Al2O3'], s=20, c="g", lw=0.25, ec='k', label='Natural')
ax[3].scatter(ks_comp_synth['SiO2'], ks_comp_synth['Al2O3'], s=20, c="r", lw=0.25, ec='k', label='Synthetic')
ax[3].set_xlabel('SiO2')
ax[3].set_ylabel('Al2O3')
plt.tight_layout()
plt.show()
